# Chapter 5: Generating Test Questions from Geometric Structure

**Learning objectives:**
- Understand the six geometric generators and how each exercises a different GMS primitive
- Generate questions with provable ground truth and verify them against the store
- See how the bridge layer converts eval questions to DOE format
- Use DOE expansion to multiply base questions into factor-varied test cases

---
## Preamble

This notebook builds on the GMS expert store created in Chapter 4. Instead of querying the store manually, we use **question generators** — classes that mine the geometric structure to produce test questions with provable ground truth.

The key layers involved:
- `knowlytix.harness.testing.generators` — six geometric generators, each exercising a different GMS primitive
- `eval.generators` — classic evaluation generators (cross-reference, multi-hop, counting)
- `knowlytix.harness.testing.generators_taxonomy` — taxonomy generators for adversarial, boundary and robustness testing
- `knowlytix.harness.testing.bridge` — adapter that converts eval questions to DOE format
- `graphdoe` — the DOE expansion engine (DesignMatrix, QuestionRephraser)

In [ ]:
# === Preamble (all notebooks) ===
import warnings
warnings.filterwarnings("ignore")

import torch
from collections import Counter

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [ ]:
from importlib.resources import files
# Load the Chapter 4 store (no retraining — loads from disk)
from knowlytix.harness.testing import DOEGMSBenchmark, DOEHarnessConfig

config = DOEHarnessConfig(
    markdown_path=str(files("knowlytix.harness.testing").joinpath("tests/model_risk_assessment.md")),
    stores_dir="gmsh_testing_ext/lab_stores",
    store_name="ch04_ground_truth",
    n_runs=4,
    ingest_epochs=500,
    device=str(DEVICE),
)

bench = DOEGMSBenchmark(config)
store = bench.ingest()

stats = store.stats()
print(f"\nStore: {stats['entities']} entities, {stats['relations']} relations, "
      f"{stats['triples']} triples, {stats['enm_entries']} ENM")

---

## The Generator Architecture

Every question generator follows the same protocol:

1. **Mine** the store for candidate facts (triples, ENM entries, paths).
2. **Construct** a natural-language question whose answer is uniquely determined by the store.
3. **Attach** a `gms_answer_fn(store)` closure that recomputes the answer at verification time.

The base class `QuestionGenerator` defines this contract:

```python
class QuestionGenerator(ABC):
    category: str = ""

    def generate(self, store) -> list[GeneratedQuestion]: ...
    def sample(self, candidates, max_questions, seed=42): ...
    def _validate(self, store, question) -> bool: ...
```

Each generator produces `GeneratedQuestion` objects with six fields: `qid`, `category`, `natural_language`, `gms_answer_fn`, `ground_truth` and `answer_type`. The `_validate` method re-runs the answer function and checks that it matches the stored ground truth — a self-consistency check that catches generator bugs before they reach the evaluation pipeline.

The six **geometric generators** each exercise a different GMS primitive:

| Generator | GMS Primitive | Question Form | Answer Type |
|-----------|--------------|---------------|-------------|
| `PlausibilityGenerator` | `score_triple()` | "Which is more plausible: A or B?" | `str` (A/B) |
| `TensionGenerator` | `tension_energy()` | "Do these entities agree, contradict or are unrelated?" | `str` |
| `HolonomyGenerator` | `check_holonomy()` | "Is the indirect path consistent?" | `bool` |
| `LinkPredictionGenerator` | `link_predict()` | "What is the X of Y?" | `str` |
| `MultiHopTransportGenerator` | `path_score()` | "Follow relation chain: what do you reach?" | `str` |
| `PhaseComparisonGenerator` | ENM `lookup_enm()` | "Which has higher value: A or B?" | `str` (A/B) |

### Running All Six Geometric Generators

Let us instantiate all six and run them against the store. The `geometric_generators()` registry function returns one instance of each.

In [ ]:
from knowlytix.harness.testing.generators import geometric_generators

geo_gens = geometric_generators()

print(f"{'Generator':30s} {'Category':22s} {'Generated':>10s} {'Valid':>8s}")
print("-" * 74)
geo_totals = {}
for gen in geo_gens:
    questions = gen.generate(store)
    valid = [q for q in questions if gen._validate(store, q)]
    geo_totals[gen.category] = (len(questions), len(valid))
    print(f"{type(gen).__name__:30s} {gen.category:22s} {len(questions):>10d} {len(valid):>8d}")

total = sum(v[0] for v in geo_totals.values())
print(f"\n{'TOTAL':30s} {'':22s} {total:>10d}")

From the 889 triples and 261 ENM entries in our store the six geometric generators produce 268 questions — all 268 pass self-validation (the answer function reproduces the stored ground truth). The generators are deterministic: given the same store and seed they always produce the same questions.

Note that each generator produces a different number of candidates. PlausibilityGenerator mines 50 triples and filters for clear geodesic separation, yielding 68. PhaseComparisonGenerator draws 40 pairs from ENM entries. The smallest pool (30) comes from MultiHopTransportGenerator because valid two-hop paths are harder to find.

---

## Geometric Generators in Detail

Let us inspect sample questions from each generator to see how GMS primitives become test questions.

### PlausibilityGenerator — Geodesic Distance on $S^{m-1}$

This generator picks a true triple $(h, r, t)$ and a random wrong tail $t'$, then asks which tail is more plausible. The ground truth comes from comparing geodesic distances: $d(R_r(v_h), v_t) < d(R_r(v_h), v_{t'})$. Only pairs with a clear margin (>0.1 radians) are kept.

In [ ]:
from knowlytix.harness.testing.generators import PlausibilityGenerator

plaus_gen = PlausibilityGenerator()
plaus_qs = plaus_gen.generate(store)
sample = plaus_gen.sample(plaus_qs, 3, seed=42)

print(f"PlausibilityGenerator: {len(plaus_qs)} questions\n")
for q in sample:
    print(f"[{q.qid}]")
    print(f"  Q: {q.natural_language}")
    print(f"  Ground truth: {q.ground_truth}")
    computed = q.gms_answer_fn(store)
    print(f"  Verified:     {computed}  (match={computed == q.ground_truth})")
    meta = q.metadata
    print(f"  Geodesic (real): {meta['geodesic_real']:.4f}, "
          f"(wrong): {meta['geodesic_wrong']:.4f}, "
          f"margin: {meta['geodesic_wrong'] - meta['geodesic_real']:.4f}")
    print()

The `in_section` triples have the widest margins (close to 1.0 radian) because section membership is the most consistent geometric signal. Relation-specific triples like `has_tier` have narrower margins (~0.2) but are still clearly separated. The metadata records both geodesic distances so we can audit the generator's decisions.

### TensionGenerator — Clifford Tension Energy

This generator picks entity pairs and classifies their relationship using tension energy thresholds:
- $E(a,b) < 0.8$ → **agree** (entities share significant geometric structure)
- $0.8 \leq E(a,b) \leq 1.6$ → **unrelated** (typical for entities in different sections)
- $E(a,b) > 1.6$ → **contradict** (opposing semantic content)

In [ ]:
from knowlytix.harness.testing.generators import TensionGenerator

tens_gen = TensionGenerator()
tens_qs = tens_gen.generate(store)
sample = tens_gen.sample(tens_qs, 3, seed=42)

print(f"TensionGenerator: {len(tens_qs)} questions\n")
for q in sample:
    print(f"[{q.qid}]")
    print(f"  Q: {q.natural_language[:120]}")
    print(f"  Ground truth: {q.ground_truth}")
    computed = q.gms_answer_fn(store)
    print(f"  Verified:     {computed}  (match={computed == q.ground_truth})")
    print(f"  Tension energy: {q.metadata['tension_energy']:.4f}")
    print()

Most entity pairs in this document fall in the "unrelated" band (tension energy 1.0–1.4). This makes sense: the model risk assessment describes 12 independent models, so cross-model entity pairs are geometrically distant but not contradictory. True contradiction requires entities with directly opposing semantics (e.g. "passes" vs. "fails").

### HolonomyGenerator — Rotor Composition Consistency

Holonomy checks whether following two successive rotors (via an intermediate entity) is consistent with the direct rotor. Given a path $h \xrightarrow{r_1} m \xrightarrow{r_2} t$ and a direct relation $h \xrightarrow{r_d} t$, the holonomy defect measures $\| R_{r_2} \circ R_{r_1} - R_{r_d} \|$. A defect near zero means the indirect path is geometrically consistent.

In [ ]:
from knowlytix.harness.testing.generators import HolonomyGenerator

holo_gen = HolonomyGenerator()
holo_qs = holo_gen.generate(store)
sample = holo_gen.sample(holo_qs, 3, seed=42)

print(f"HolonomyGenerator: {len(holo_qs)} questions\n")
for q in sample:
    m = q.metadata
    print(f"[{q.qid}]")
    print(f"  Path: {m['head']} --[{m['r1']}]--> {m['mid']} --[{m['r2']}]--> {m['tail']}")
    print(f"  Direct relation: {m['r_direct']}")
    print(f"  Holonomy defect: {m['holonomy_defect']:.4f}")
    print(f"  Consistent? {q.ground_truth}")
    computed = q.gms_answer_fn(store)
    print(f"  Verified:   {computed}  (match={str(computed) == str(q.ground_truth)})")
    print()

The holonomy defects range from 0.5 to 1.3 — all exceeding the consistency threshold, so all these indirect paths are flagged as inconsistent. This is expected: the `has_downstream_model` relation connects different models, so composing it with `in_section` or `has_risk_tier` does not reproduce the direct relation's rotor. Holonomy consistency is a strong signal for structural integrity in the knowledge graph.

### LinkPredictionGenerator — Geometric Beam Search

This generator asks "What is the X of Y?" and verifies the answer using the GMS link prediction engine, which ranks candidate tails by geodesic proximity to the rotor-transported head $R_r(v_h)$.

In [ ]:
from knowlytix.harness.testing.generators import LinkPredictionGenerator

link_gen = LinkPredictionGenerator()
link_qs = link_gen.generate(store)
sample = link_gen.sample(link_qs, 3, seed=42)

print(f"LinkPredictionGenerator: {len(link_qs)} questions\n")
for q in sample:
    m = q.metadata
    print(f"[{q.qid}]")
    print(f"  Q: {q.natural_language}")
    print(f"  Ground truth: {q.ground_truth}")
    computed = q.gms_answer_fn(store)
    print(f"  Verified:     {computed}  (match={computed == q.ground_truth})")
    # Show top-3 predictions with geodesic scores
    print("  Top predictions:")
    for entity, score in m["top_predictions"]:
        marker = " <-- correct" if entity == q.ground_truth else ""
        print(f"    {entity:30s}  geodesic={score:.4f}{marker}")
    print()

The top-3 predictions show the geometric ranking: the correct tail often ranks 2nd or 3rd because the self-entity (MDL-00X) is always closest geometrically to its own rotor-transported position. The generator filters for questions where the correct answer appears in the top-3 candidates.

### MultiHopTransportGenerator — Two-Hop Relational Paths

This generator constructs two-hop queries: "Starting from entity A, follow relation $r_1$ to reach B, then follow $r_2$. What do you reach?" The path geodesic score measures how well the composed transport $R_{r_2}(R_{r_1}(v_A))$ matches the final entity.

In [ ]:
from knowlytix.harness.testing.generators import MultiHopTransportGenerator

mhop_gen = MultiHopTransportGenerator()
mhop_qs = mhop_gen.generate(store)
sample = mhop_gen.sample(mhop_qs, 3, seed=42)

print(f"MultiHopTransportGenerator: {len(mhop_qs)} questions\n")
for q in sample:
    m = q.metadata
    print(f"[{q.qid}]")
    print(f"  Path: {m['head']} --[{m['r1']}]--> {m['mid']} --[{m['r2']}]--> {m['tail']}")
    print(f"  Q: {q.natural_language[:120]}")
    print(f"  Ground truth: {q.ground_truth}")
    computed = q.gms_answer_fn(store)
    print(f"  Verified:     {computed}  (match={computed == q.ground_truth})")
    print(f"  Path geodesic: {m['path_geodesic']:.4f}")
    print()

The multi-hop paths typically go through model entities: "Starting from an action item, follow `has_model_id` to MDL-001, then follow `has_approval_rate`." Path geodesic scores of 1.3–1.7 indicate moderate geometric alignment — the composed rotor transport is approximate but sufficient for the ground truth to be uniquely recoverable.

### PhaseComparisonGenerator — ENM Numeric Comparisons

This generator draws two ENM entries from the same section and asks which has the higher value. The ground truth comes from the exact IEEE 754 values in the ENM register — no geometric approximation involved.

In [ ]:
from knowlytix.harness.testing.generators import PhaseComparisonGenerator

phase_gen = PhaseComparisonGenerator()
phase_qs = phase_gen.generate(store)
sample = phase_gen.sample(phase_qs, 3, seed=42)

print(f"PhaseComparisonGenerator: {len(phase_qs)} questions\n")
for q in sample:
    m = q.metadata
    print(f"[{q.qid}]")
    print(f"  Q: {q.natural_language[:120]}")
    print(f"  Ground truth: {q.ground_truth}")
    computed = q.gms_answer_fn(store)
    print(f"  Verified:     {computed}  (match={computed == q.ground_truth})")
    print(f"  ENM section: {m['category']}")
    print(f"  Value A: {m['value_a']:.4f} ({m['entity_a'][:40]})")
    print(f"  Value B: {m['value_b']:.4f} ({m['entity_b'][:40]})")
    print()

The PhaseComparisonGenerator is the only geometric generator that uses the ENM register rather than the hypersphere embeddings. This makes it complementary to the other five: it tests whether an agent can reason about exact numeric values, while the others test geometric understanding. Together, the six generators cover both the continuous (geometric) and discrete (ENM) aspects of the GMS.

---

## Lab 5.1 — Generate and Inspect Test Questions

Let us now use the `geometric_generators()` registry to generate questions, verify them and examine the full question anatomy.

In [ ]:
# Lab 5.1: Generate, sample and verify geometric questions
from knowlytix.harness.testing.generators import geometric_generators

all_geo_qs = []
for gen in geometric_generators():
    questions = gen.generate(store)
    sampled = gen.sample(questions, max_questions=3, seed=42)
    # Verify each sampled question
    for q in sampled:
        computed = q.gms_answer_fn(store)
        assert str(computed) == str(q.ground_truth), \
            f"Verification failed for {q.qid}: {computed} != {q.ground_truth}"
    all_geo_qs.extend(sampled)

print(f"Sampled {len(all_geo_qs)} questions (3 per generator x 6 generators)")
print(f"\n{'QID':35s} {'Category':22s} {'Answer':>10s} {'Type':>6s}")
print("-" * 77)
for q in all_geo_qs:
    gt = str(q.ground_truth)[:10]
    print(f"{q.qid:35s} {q.category:22s} {gt:>10s} {q.answer_type:>6s}")

In [11]:
# Anatomy of a single question
q = all_geo_qs[0]
print("=== Question Anatomy ===")
print(f"  qid:             {q.qid}")
print(f"  category:        {q.category}")
print(f"  answer_type:     {q.answer_type}")
print(f"  natural_language: {q.natural_language}")
print(f"  ground_truth:    {q.ground_truth}")
print(f"  llm_prompt:      {q.llm_prompt[:150]}...")
print(f"  gms_answer_fn:   {q.gms_answer_fn}")
print(f"  metadata keys:   {list(q.metadata.keys())}")
print("\nFull metadata:")
for k, v in q.metadata.items():
    print(f"    {k}: {v}")

=== Question Anatomy ===
  qid:             plausibility_cmp_014
  category:        plausibility
  answer_type:     str
  natural_language: Which entity is more plausible as the 'has_detail' of 'Chief Model Risk Officer': (A) Dr. Elena Vasquez or (B) Doc classification feeds AML alerts?
  ground_truth:    A
  llm_prompt:      Given the document, which entity is more plausible as the 'has_detail' of 'Chief Model Risk Officer'?
  (A) Dr. Elena Vasquez
  (B) Doc classification...
  gms_answer_fn:   <function PlausibilityGenerator.generate.<locals>._answer_fn at 0xfda34b84cea0>
  metadata keys:   ['head', 'relation', 'true_tail', 'wrong_tail', 'geodesic_real', 'geodesic_wrong']

Full metadata:
    head: Chief Model Risk Officer
    relation: has_detail
    true_tail: Dr. Elena Vasquez
    wrong_tail: Doc classification feeds AML alerts
    geodesic_real: 1.0556
    geodesic_wrong: 1.5699


Every question carries its full provenance: which entities are involved, the geodesic scores that determine the ground truth and the answer function closure that can recompute the answer at any time. This makes each question independently auditable — you can trace every ground truth back to a specific GMS primitive and verify it.

---

## Beyond Geometric: Classic and Taxonomy Generators

The geometric generators are only one family. The GMSH framework includes three additional families:

| Family | Generators | Focus |
|--------|-----------|-------|
| **Classic eval** | ExactRecall, Threshold, CrossReference, Contradiction, MultiHop, Counting | Standard NLP evaluation tasks |
| **Extended** | ExactRetrieval, Negative, WorkflowCompliance | GMS-native tasks |
| **Taxonomy** | Boundary, Adversarial, Robustness, RecoveryTest, Governance | Stress-testing and safety |

Let us run all of them and compare the question pools.

In [ ]:
# Classic eval generators
from knowlytix.benchmark.eval.generators import default_generators

classic_gens = default_generators()
print("=== Classic Eval Generators ===")
print(f"{'Generator':30s} {'Category':20s} {'Generated':>10s} {'Valid':>8s}")
print("-" * 72)
for gen in classic_gens:
    qs = gen.generate(store)
    valid = [q for q in qs if gen._validate(store, q)]
    print(f"{type(gen).__name__:30s} {gen.category:20s} {len(qs):>10d} {len(valid):>8d}")
    if valid:
        q = valid[0]
        print(f"  Sample: {q.natural_language[:90]}")
        print(f"  GT: {str(q.ground_truth)[:60]}")

In [ ]:
# Extended generators
from knowlytix.harness.testing.test_generators import extended_generators

ext_gens = extended_generators()
print("=== Extended Generators ===")
print(f"{'Generator':30s} {'Category':20s} {'Generated':>10s} {'Valid':>8s}")
print("-" * 72)
for gen in ext_gens:
    qs = gen.generate(store)
    valid = [q for q in qs if gen._validate(store, q)]
    print(f"{type(gen).__name__:30s} {gen.category:20s} {len(qs):>10d} {len(valid):>8d}")
    if valid:
        q = valid[0]
        print(f"  Sample: {q.natural_language[:90]}")
        print(f"  GT: {str(q.ground_truth)[:60]}")

In [ ]:
# Taxonomy generators
from knowlytix.harness.testing.generators_taxonomy import taxonomy_generators

tax_gens = taxonomy_generators()
print("=== Taxonomy Generators ===")
print(f"{'Generator':30s} {'Category':20s} {'Generated':>10s} {'Valid':>8s}")
print("-" * 72)
for gen in tax_gens:
    qs = gen.generate(store)
    valid = [q for q in qs if gen._validate(store, q)]
    print(f"{type(gen).__name__:30s} {gen.category:20s} {len(qs):>10d} {len(valid):>8d}")
    if valid:
        q = valid[0]
        print(f"  Sample: {q.natural_language[:90]}")
        print(f"  GT: {str(q.ground_truth)[:60]}")

Several things to note:

- **CrossReferenceGenerator** is the most prolific (1,332 questions) because it mines all relation pairs — a combinatorial explosion. The DOE sampling step (the Bridge section below) trims this down to 10 per category.
- **ExactRecall** and **Threshold** generators produce zero questions from this document because they require specific document structures not present in the model risk assessment.
- **NegativeGenerator** produces 30 candidates but none pass validation — negative questions ("Is X false?") are harder to verify automatically and are typically validated by a human.
- **Taxonomy generators** (Boundary, Adversarial, Robustness, RecoveryTest, Governance) all produce valid questions. Their question text includes adversarial perturbations: typos in entity names (Robustness), contradictory context hints (Adversarial), simulated tool failures (RecoveryTest) and tool-restriction constraints (Governance).

---

## The Bridge: From Eval Questions to DOE Format

The geometric generators produce `GeneratedQuestion` objects from the `eval.generators.base` module. These carry a `gms_answer_fn(store)` closure. However, the DOE expansion engine (graphdoe) expects `FSBQuestion` objects from `finstructbench.generators.base`, which carry a `graph_answer_fn(graph)` closure.

The **bridge** module (`knowlytix.harness.testing.bridge`) adapts between these two formats:

```python
def adapt_question(eq: EvalQuestion, store) -> FSBQuestion:
    return FSBQuestion(
        qid=eq.qid, category=eq.category,
        natural_language=eq.natural_language,
        graph_answer_fn=lambda _graph: eq.gms_answer_fn(store),
        ground_truth=eq.ground_truth, ...
    )
```

The key trick: the closure captures `store` and ignores the `graph` argument. This lets the DOE machinery treat GMS-backed questions identically to graph-based ones.

The `generate_base_questions()` function runs all generators, validates results and returns adapted questions:

In [ ]:
# Bridge: generate base questions (geometric only)
from knowlytix.harness.testing.bridge import generate_base_questions

geo_names = ["plausibility", "tension", "holonomy",
             "link_prediction", "multi_hop_transport", "phase_comparison"]

base_geo = generate_base_questions(
    store, generator_names=geo_names, max_per_category=10, seed=42
)

print(f"\nBase questions (geometric only): {len(base_geo)}")
cat_counts = Counter(q.category for q in base_geo)
print(f"\n{'Category':25s} {'Count':>6s}")
print("-" * 33)
for cat, cnt in cat_counts.most_common():
    print(f"{cat:25s} {cnt:>6d}")

# Verify the adapted answer function works (graph arg is ignored)
q0 = base_geo[0]
computed = q0.graph_answer_fn(None)
print(f"\nVerification: {q0.qid} -> {computed} (expected {q0.ground_truth})")

In [16]:
# Bridge: generate ALL base questions (no filter)
all_base = generate_base_questions(store, generator_names=None, max_per_category=10, seed=42)

print(f"\nAll base questions: {len(all_base)}")
all_cats = Counter(q.category for q in all_base)
print(f"\n{'Category':25s} {'Count':>6s}")
print("-" * 33)
for cat, cnt in sorted(all_cats.items()):
    print(f"{cat:25s} {cnt:>6d}")
print(f"\n{'TOTAL':25s} {len(all_base):>6d} ({len(all_cats)} categories)")

  Base questions: 160 (20 generators, max 10/cat)

All base questions: 160

Category                   Count
---------------------------------
adversarial                   10
boundary                      10
counting                      10
cross_reference               10
exact_retrieval               10
governance                    10
holonomy                      10
link_prediction               10
multi_hop                     10
multi_hop_transport           10
phase_comparison              10
plausibility                  10
recovery_test                 10
robustness                    10
tension                       10
workflow_compliance           10

TOTAL                        160 (16 categories)


The bridge caps each category at 10 questions (via `max_per_category`), turning the 2,000+ raw candidates into a balanced set of 160 base questions across 16 categories. This balancing is important: without it, CrossReference (1,332 candidates) would dominate the test suite while less prolific generators would be underrepresented.

---

## The DOE Factor System

Before we expand base questions into DOE test cases, we need to understand the **factor system**. DOE factors are experimental variables that modify how a question is presented — without changing the ground truth. The `graphdoe.FactorLoader` manages a catalog of 40 factors organized into 10 families and 13 preset groups.

In [ ]:
from knowlytix.harness.graphdoe import FactorLoader

fl = FactorLoader()

# Factor families (10 families, 40 factors total)
families = fl.list_factors()
total_factors = sum(len(v) for v in families.values())
print(f"=== Factor Catalog: {total_factors} factors in {len(families)} families ===\n")
for family, names in families.items():
    print(f"  {family:25s} ({len(names):>2d}): {names}")

In [18]:
# Preset groups
groups = sorted(fl.list_groups())
print(f"=== Preset Groups ({len(groups)}) ===\n")
for gn in groups:
    factors = fl.load_by_group(gn)
    names = [f["name"] for f in factors]
    print(f"  {gn:20s} ({len(factors):>2d} factors): {names}")

=== Preset Groups (13) ===

  adversarial          ( 3 factors): ['instruction_conflict', 'answer_anchoring', 'confidence_pressure']
  agentic              ( 5 factors): ['tool_availability', 'time_pressure', 'iteration_limit', 'reasoning_cue', 'hop_count']
  comprehensive        (20 factors): ['clarity', 'length', 'specificity', 'answer_format', 'role', 'expertise', 'style', 'reasoning_cue', 'context_scope', 'distractor_content', 'entity_aliasing', 'numeric_format', 'instruction_conflict', 'answer_anchoring', 'confidence_pressure', 'paraphrase_depth', 'noise', 'complexity', 'emotional_frame', 'intent_count']
  context_stress       ( 4 factors): ['context_scope', 'section_hint', 'distractor_content', 'context_order']
  conversational       ( 3 factors): ['memory', 'history', 'turn_position']
  entity_stress        ( 4 factors): ['entity_aliasing', 'numeric_format', 'unit_variation', 'precision_hint']
  full                 (40 factors): ['query_type', 'clarity', 'length', 'specificity'

In [19]:
# The quick_screen group: 3 factors, ideal for demos
qs_factors = fl.load_by_group("quick_screen")
print("=== quick_screen factors ===\n")
for f in qs_factors:
    print(f"  {f['name']:25s} ({f['cardinality']} levels): {f['categories']}")
    print(f"    {f['description']}")

=== quick_screen factors ===

  clarity                   (3 levels): ['Clear', 'Ambiguous', 'Misleading']
    Query clarity level
  entity_aliasing           (3 levels): ['Exact', 'Synonym', 'Abbreviated']
    How entity names are presented in the question
  reasoning_cue             (3 levels): ['Direct', 'Step_By_Step', 'No_Cue']
    Chain-of-thought / prompting strategy


The `quick_screen` group uses just three factors — clarity, entity aliasing and reasoning cue — each with three levels (27 possible combinations). This is enough to detect major sensitivity patterns without requiring thousands of test runs. In Chapter 3 we introduced the design matrix; now we will see it in action.

---

## Lab 5.2 — DOE Expansion

### The Design Matrix

A `DesignMatrix` generates a space-filling sample from the factor space using Sobol sequences. With 3 factors × 3 levels each and `n_runs=4`, we get a 4×3 matrix where each row is a factor assignment.

In [ ]:
from knowlytix.harness.graphdoe import DesignMatrix

dm = DesignMatrix(qs_factors, n_runs=4, seed=42)
print(f"Design matrix: {dm.n_runs} runs x {len(qs_factors)} factors")
print(f"Method: {dm.method}\n")
print(dm.matrix.to_string())

Each row is a unique combination of factor levels. Run 3 (`Clear`, `Exact`, `Direct`) is the "clean" baseline — no ambiguity, no aliasing, straightforward prompting. Run 1 (`Misleading`, `Synonym`, `Step_By_Step`) is the most adversarial — the question text is deliberately misleading, entity names are paraphrased and a chain-of-thought cue is injected.

### Expanding Base Questions

The `QuestionRephraser` takes each base question and modifies its text according to the factor assignment. The ground truth is preserved — only the presentation changes.

In [ ]:
from knowlytix.harness.graphdoe import QuestionRephraser

# Expand 3 plausibility questions x 4 runs = 12 DOE questions
base_plaus = generate_base_questions(
    store, generator_names=["plausibility"], max_per_category=3, seed=42
)

qr = QuestionRephraser(method="template")
doe_plaus = qr.expand(base_plaus, dm.matrix)

print(f"Expansion: {len(base_plaus)} base x {dm.n_runs} runs = {len(doe_plaus)} DOE questions")

# Show one base question across all 4 runs
base_id = base_plaus[0].qid
variants = [dq for dq in doe_plaus if dq.base_qid == base_id]

print(f"\n--- '{base_id}' across {len(variants)} runs ---")
print(f"  Original: {base_plaus[0].natural_language}\n")
for v in variants:
    print(f"  {v.doe_qid}:")
    print(f"    Factors: {v.factor_assignment}")
    print(f"    NL:      {v.question.natural_language}")
    print(f"    GT:      {v.question.ground_truth}")
    print()

Notice how the template rephraser modifies the question:

- **Run 0** (Ambiguous/Abbreviated/No_Cue): prepends `[Ambiguous]` tag
- **Run 1** (Misleading/Synonym/Step_By_Step): prepends "Some sources suggest a different figure, but..." and lowercases entity names
- **Run 2** (Ambiguous/Synonym/No_Cue): prepends `[Ambiguous]` tag
- **Run 3** (Clear/Exact/Direct): no modification — the original question unchanged

The ground truth remains the same across all four runs. This is the core DOE principle: by systematically varying presentation factors while holding the answer constant, we can measure how much each factor affects agent accuracy.

### Full DOE Expansion via DOEGMSBenchmark

The `DOEGMSBenchmark.generate_questions()` method runs the full pipeline: all generators → bridge → sampling → DOE expansion.

In [22]:
# Full DOE expansion via the benchmark harness
doe_questions = bench.generate_questions()

print(f"DOE questions: {len(doe_questions)}")
doe_cats = Counter(dq.question.category for dq in doe_questions)
print(f"\n{'Category':25s} {'Base':>6s} {'x Runs':>7s} {'= DOE':>7s}")
print("-" * 48)
for cat, cnt in sorted(doe_cats.items()):
    base_cnt = cnt // config.n_runs
    print(f"{cat:25s} {base_cnt:>6d} {'x ' + str(config.n_runs):>7s} {cnt:>7d}")
print(f"\n{'TOTAL':25s} {len(doe_questions) // config.n_runs:>6d} {'':>7s} {len(doe_questions):>7d}")

  Base questions: 160 (20 generators, max 10/cat)

  DOE expansion: 160 base x 4 runs = 640 questions
  Factors: ['clarity', 'entity_aliasing', 'reasoning_cue']
  [BackgroundTrain] Starting: 10 entities, 21 relations, 25 triples


  [BackgroundTrain] Done: Trained: 10 entities, 21 relations, 50 steps in 4.0s
  Store saved to doe_tracing_store/_doe_tracing
  [BackgroundTrain] Store saved
DOE questions: 640

Category                    Base  x Runs   = DOE
------------------------------------------------
adversarial                   10     x 4      40
boundary                      10     x 4      40
counting                      10     x 4      40
cross_reference               10     x 4      40
exact_retrieval               10     x 4      40
governance                    10     x 4      40
holonomy                      10     x 4      40
link_prediction               10     x 4      40
multi_hop                     10     x 4      40
multi_hop_transport           10     x 4      40
phase_comparison              10     x 4      40
plausibility                  10     x 4      40
recovery_test                 10     x 4      40
robustness                    10     x 4      40
tension                       10     

The multiplier effect is clear: 160 base questions × 4 runs = 640 DOE questions, each with a unique factor assignment. This is the power of DOE-based testing — from a single ingested document we generate a systematic, balanced test suite that probes multiple dimensions of agent behavior simultaneously.

### Inspecting a DOE Question

Each `DOEQuestion` wraps the rephrased question, the factor assignment, and links back to its base question:

In [23]:
# Inspect a single DOE question
dq = doe_questions[0]
print("=== DOE Question Anatomy ===")
print(f"  doe_qid:          {dq.doe_qid}")
print(f"  base_qid:         {dq.base_qid}")
print(f"  factor_assignment: {dq.factor_assignment}")
print("\n  Wrapped question:")
print(f"    category:        {dq.question.category}")
print(f"    natural_language: {dq.question.natural_language[:120]}")
print(f"    ground_truth:    {dq.question.ground_truth}")
print(f"    answer_type:     {dq.question.answer_type}")

=== DOE Question Anatomy ===
  doe_qid:          cross_reference_1309_run000
  base_qid:         cross_reference_1309
  factor_assignment: {'clarity': 'Ambiguous', 'entity_aliasing': 'Abbreviated', 'reasoning_cue': 'No_Cue'}

  Wrapped question:
    category:        cross_reference
    natural_language: [Ambiguous] Which entities appear in both the 'has_annual_loss_exposure' and 'has_delta_auc' relationships?
    ground_truth:    {'MDL-007', 'MDL-011', 'MDL-002', 'MDL-005', 'MDL-012', 'MDL-003', 'MDL-001', 'MDL-004', 'MDL-009', 'MDL-010', 'MDL-006', 'MDL-008'}
    answer_type:     set_str


---

## Exercises

### Exercise 5.1

Run the `PlausibilityGenerator` with the full candidate pool (no sampling). Plot a histogram of the geodesic margin (`geodesic_wrong - geodesic_real`) for all 68 questions. What is the median margin? Are there any questions with margins below 0.2 radians?

### Exercise 5.2

Use `generate_base_questions()` with the `comprehensive` factor group (20 factors) and `n_runs=8`. How many DOE questions does this produce? What is the ratio of DOE questions to base questions? Compute the factor coverage using `dm.coverage_report()`.

### Exercise 5.3

Write a function that takes a list of DOE questions and returns a DataFrame with columns `[doe_qid, category, clarity, entity_aliasing, reasoning_cue, ground_truth]`. Use it to verify that every (category, factor_assignment) combination appears at most once in the design.

In [24]:
# === Exercise 5.1: Your code here ===
# Starter: uncomment and run
# plaus_gen = PlausibilityGenerator()
# all_plaus = plaus_gen.generate(store)
# margins = [q.metadata["geodesic_wrong"] - q.metadata["geodesic_real"] for q in all_plaus]
# import matplotlib.pyplot as plt
# plt.hist(margins, bins=20)
# plt.xlabel("Geodesic margin (radians)")
# plt.title(f"Plausibility margins (n={len(margins)}, median={np.median(margins):.3f})")
# plt.show()

In [25]:
# === Exercise 5.2: Your code here ===


In [26]:
# === Exercise 5.3: Your code here ===


---

## Summary

In this chapter we explored the question generation pipeline that turns a GMS expert store into a systematic test suite:

1. **Six geometric generators** mine different GMS primitives (geodesic scoring, tension energy, holonomy, link prediction, multi-hop transport, phase comparison) to produce 268 questions with provable ground truth — 100% pass self-validation.
2. **Classic, extended and taxonomy generators** add 16 categories covering standard NLP evaluation, adversarial perturbations, robustness and governance constraints.
3. **The bridge** adapts eval-format questions to DOE-format questions via a closure that captures the store reference, enabling seamless interoperability between the two question types.
4. **The factor catalog** (40 factors in 10 families, 13 preset groups) defines how questions are systematically varied. The `quick_screen` group uses 3 factors (clarity, entity aliasing, reasoning cue) for lightweight screening.
5. **DOE expansion** multiplies 160 base questions × 4 Sobol-sampled runs = 640 DOE questions, each with a unique factor assignment. The ground truth is preserved across all runs — only the presentation changes.
6. **The multiplier effect** means a single ingested document yields hundreds of test cases that systematically probe different dimensions of agent behavior.

**Next:** Chapter 6 shows how the GMS geometric judge scores agent responses against these ground truths — turning raw answers into pass/fail verdicts with continuous confidence scores.